In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# !MAX_JOBS=4 TORCH_CUDA_ARCH_LIST="7.5" CAUSAL_CONV1D_FORCE_BUILD=TRUE \
#   pip wheel causal-conv1d --no-build-isolation --no-deps -w /kaggle/working/wheels

In [ ]:
import glob
whl = glob.glob("/kaggle/input/**/causal_conv1d-*.whl", recursive=True)
print(whl)

In [2]:
!pip install --no-deps {whl[0]}
!pip install -U transformers peft trl accelerate datasets flash-linear-attention

Processing /kaggle/input/datasets/erygfry/causal-conv1d-t4/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl
causal-conv1d is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [3]:
!python -c "import sys, torch; print(sys.version); print(torch.__version__, 'cuda', torch.version.cuda)"
!nvcc --version
!nproc; free -h

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
2.11.0+cu128 cuda 12.8
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
4
               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.6Gi        18Gi       1.7Mi        12Gi        29Gi
Swap:             0B          0B          0B


In [5]:
!pip show causal-conv1d

Name: causal_conv1d
Version: 1.7.0
Summary: Causal depthwise conv1d in CUDA, with a PyTorch interface
Home-page: https://github.com/Dao-AILab/causal-conv1d
Author: Tri Dao
Author-email: tri@tridao.me
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: ninja, packaging, torch
Required-by: 


In [7]:
import torch, torch.nn.functional as F
from causal_conv1d import causal_conv1d_fn

B, D, T, W = 2, 64, 128, 4
x = torch.randn(B, D, T, device="cuda", dtype=torch.float16)
w = torch.randn(D, W, device="cuda", dtype=torch.float16)
b = torch.randn(D, device="cuda", dtype=torch.float16)

out = causal_conv1d_fn(x, w, b, activation="silu")
# эталон: depthwise-свёртка с паддингом слева, обрезаем хвост — так она становится причинной
ref = F.silu(F.conv1d(x.float(), w.float().unsqueeze(1), b.float(),
                      padding=W - 1, groups=D)[..., :T])
print("max diff:", (out.float() - ref).abs().max().item())

max diff: 0.0038042068481445312


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.qwen3_5 import modeling_qwen3_5 as mq
print("fast path:", getattr(mq, "is_fast_path_available", "флага нет"))

name = "Qwen/Qwen3.5-0.8B-Base"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).cuda().eval()
# в старых версиях transformers вместо dtype= было torch_dtype=

# какие реализации реально лежат внутри слоёв
seen = set()
for m in model.modules():
    for attr, val in vars(m).items():
        if callable(val) and ("conv1d" in attr or "delta_rule" in attr):
            seen.add((attr, getattr(val, "__module__", "?")))
for s in sorted(seen):
    print(s)

fast path: флага нет


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [10]:
# 1. Что transformers сам думает про fla
from transformers.utils import import_utils
print("fla available:", import_utils.is_flash_linear_attention_available())

# 2. Импортируется ли fla напрямую — если упадёт, трейсбек и есть ответ
from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule
print("fla import ok")

# 3. Что лежит в модуле модели под этими именами
for n in dir(mq):
    if any(s in n.lower() for s in ("delta", "conv1d", "fla", "fast")):
        print(n, "->", getattr(mq, n))

fla available: True
fla import ok
FlashAttentionKwargs -> <class 'transformers.modeling_flash_attention_utils.FlashAttentionKwargs'>
Qwen3_5GatedDeltaNet -> <class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5GatedDeltaNet'>
causal_conv1d_fn -> <function causal_conv1d_fn at 0x7ce878425a80>
causal_conv1d_update -> <function causal_conv1d_update at 0x7ce878425940>
is_flash_attention_requested -> <function is_flash_attention_requested at 0x7ce87a356160>
torch_chunk_gated_delta_rule -> <function torch_chunk_gated_delta_rule at 0x7ce878425d00>
torch_recurrent_gated_delta_rule -> <function torch_recurrent_gated_delta_rule at 0x7ce878425ee0>


In [11]:
import random
text = "Фильм оставил очень сильное впечатление: прекрасная игра актёров, продуманный сюжет и отличная музыка."
words = text.split(); random.seed(0); random.shuffle(words)

for t in [text, " ".join(words)]:
    enc = tok(t, return_tensors="pt").to("cuda")
    with torch.no_grad():
        loss = model(**enc, labels=enc.input_ids).loss
    print(f"{loss.item():.3f}  {t[:50]}")

enc = tok("Отзыв: Фильм оставил", return_tensors="pt").to("cuda")
print(tok.decode(model.generate(**enc, max_new_tokens=40, do_sample=False)[0]))

2.592  Фильм оставил очень сильное впечатление: прекрасна
5.580  оставил и сюжет прекрасная отличная очень сильное 
Отзыв: Фильм оставил меня в шоке.
Отзыв: Фильм оставил меня в шоке.
Отзыв: Фильм оставил меня в шоке.
Отзыв: Фильм оставил меня


In [12]:
from torch.profiler import profile, ProfilerActivity

enc = tok(text, return_tensors="pt").to("cuda")
with torch.no_grad(), profile(activities=[ProfilerActivity.CUDA]) as prof:
    model(**enc)

names = sorted({e.key for e in prof.key_averages()})
hits = [n for n in names if any(s in n.lower() for s in ("conv1d", "delta", "chunk", "fla"))]
print(len(names), "разных ядер")
for n in hits:
    print(n[:120])

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


70 разных ядер
chunk_fwd_kernel_o
chunk_gated_delta_rule_fwd_kernel_h_blockdim64
chunk_gated_delta_rule_fwd_kkt_solve_kernel
chunk_local_cumsum_scalar_kernel
void causal_conv1d_channellast_fwd_kernel<Causal_conv1d_channellast_fwd_kernel_traits<128, 4, 64, true, c10::Half, c10::
